<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Система способов оплаты</h2>

----

### Вариант задания № 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Разработка системы различных способов оплаты на базе класса PaymentMethod. Система предусматривает онлайн-оплату, банковский перевод и оплату наличными.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте простое, сложное и множественное наследование

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
// Интерфейс для отправки чека
public interface IReceiptable
{
    void SendReceipt(string email);
}

// Интерфейс для возврата средств
public interface IRefundable
{
    void Refund(decimal amount);
}

// Базовый класс
public class PaymentMethod
{
    private int _paymentMethodId;
    private string _methodName;
    private decimal _minAmount;
    private decimal _commission;

    public int PaymentMethodId
    {
        get { return _paymentMethodId; }
        set { _paymentMethodId = value; }
    }

    public string MethodName
    {
        get { return _methodName; }
        set { _methodName = value; }
    }

    public decimal MinAmount
    {
        get { return _minAmount; }
        set { _minAmount = value; }
    }

    public decimal Commission
    {
        get { return _commission; }
        set { _commission = value; }
    }

    public string Currency { get; set; }

    public PaymentMethod(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission)
    {
        PaymentMethodId = paymentMethodId;
        MethodName = methodName;
        MinAmount = minAmount;
        Commission = commission;
        Currency = "RUB";
    }

    public bool CheckMinimumAmount(decimal amount)
    {
        return amount >= MinAmount;
    }

    public decimal CalculateCommission(decimal amount)
    {
        return amount * (Commission / 100);
    }

    public void CancelPayment()
    {
        Console.WriteLine($"Платеж по способу '{MethodName}' отменен.");
    }

    public virtual void ProcessPayment(decimal amount)
    {
        if (CheckMinimumAmount(amount))
        {
            decimal commission = CalculateCommission(amount);

            Console.WriteLine(
                $"Успешно: платеж на сумму {amount} {Currency} " +
                $"(комиссия: {commission} {Currency}) проведен через '{MethodName}'.");
        }
        else
        {
            Console.WriteLine(
                $"Ошибка: сумма {amount} {Currency} меньше минимальной " +
                $"({MinAmount} {Currency}) для '{MethodName}'.");
        }
    }

    public virtual void GetPaymentDetails()
    {
        Console.WriteLine(
            $"ID способа: {PaymentMethodId} | Название: {MethodName} | " +
            $"Мин. сумма: {MinAmount} {Currency} | Комиссия: {Commission}%");
    }
}

// Онлайн-оплата
public class OnlinePayment : PaymentMethod, IReceiptable
{
    private string _paymentUrl;

    public string PaymentUrl
    {
        get { return _paymentUrl; }
        set { _paymentUrl = value; }
    }

    public OnlinePayment(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string paymentUrl)
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        PaymentUrl = paymentUrl;
    }

    public void SendReceipt(string email)
    {
        Console.WriteLine($"Электронный чек отправлен на почту: {email}");
    }
}

// Банковский перевод
public class BankTransfer : PaymentMethod
{
    private string _bankData;

    public string BankData
    {
        get { return _bankData; }
        set { _bankData = value; }
    }

    public BankTransfer(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string bankData)
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        BankData = bankData;
    }
}

// Оплата наличными
public class CashPayment : PaymentMethod
{
    private string _cashPickupPoint;

    public string CashPickupPoint
    {
        get { return _cashPickupPoint; }
        set { _cashPickupPoint = value; }
    }

    public CashPayment(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string cashPickupPoint)
        : base(paymentMethodId, methodName, minAmount, commission)
    {
        CashPickupPoint = cashPickupPoint;
    }
}

// Оплата банковской картой
public class CardPayment : OnlinePayment, IRefundable
{
    public string CardNumber { get; set; }
    public string CardHolder { get; set; }

    public CardPayment(int paymentMethodId, string methodName,
        decimal minAmount, decimal commission, string paymentUrl,
        string cardNumber, string cardHolder)
        : base(paymentMethodId, methodName, minAmount, commission, paymentUrl)
    {
        CardNumber = cardNumber;
        CardHolder = cardHolder;
    }

    public void Refund(decimal amount)
    {
        Console.WriteLine(
            $"Оформлен возврат {amount} {Currency} на карту {CardNumber} ({CardHolder})");
    }
}

// Класс заказа
public class Order
{
    public int OrderId { get; set; }
    public decimal TotalAmount { get; set; }

    public Order(int orderId, decimal totalAmount)
    {
        OrderId = orderId;
        TotalAmount = totalAmount;
    }

    public void Pay(PaymentMethod paymentMethod)
    {
        Console.WriteLine(
            $"\nПопытка оплаты заказа №{OrderId} на сумму {TotalAmount} руб.");

        paymentMethod.ProcessPayment(TotalAmount);
    }
}


// Создание объектов
BankTransfer bank = new BankTransfer(
    1, "Банковский перевод", 500, 1.5m,
    "ПАО Сбербанк, Р/С 407...");

CashPayment cash = new CashPayment(
    2, "Наличные", 50, 0m,
    "Касса №1 (ул. Мира, 10)");

CardPayment card = new CardPayment(
    3, "Банковская карта онлайн", 100, 2.0m,
    "https://securepay.ru",
    "2200-****-****-1234",
    "Иван Иванов");

// Вывод информации
Console.WriteLine("--- Информация о способах оплаты ---");

bank.GetPaymentDetails();
Console.WriteLine($"Реквизиты: {bank.BankData}\n");

cash.GetPaymentDetails();
Console.WriteLine($"Точка оплаты: {cash.CashPickupPoint}\n");

card.GetPaymentDetails();
Console.WriteLine(
    $"URL: {card.PaymentUrl}, Карта: {card.CardNumber}, Держатель: {card.CardHolder}\n");

// Проверка дополнительных методов
Console.WriteLine("--- Проверка дополнительных методов ---");

card.SendReceipt("student@mail.ru");
card.Refund(150);
card.CancelPayment();

// Взаимодействие объектов
Console.WriteLine("\n--- Взаимодействие объектов с классом Order ---");

Order order1 = new Order(101, 1000);
Order order2 = new Order(102, 30);

order1.Pay(card);
order2.Pay(card);

--- Информация о способах оплаты ---
ID способа: 1 | Название: Банковский перевод | Мин. сумма: 500 RUB | Комиссия: 1.5%
Реквизиты: ПАО Сбербанк, Р/С 407...

ID способа: 2 | Название: Наличные | Мин. сумма: 50 RUB | Комиссия: 0%
Точка оплаты: Касса №1 (ул. Мира, 10)

ID способа: 3 | Название: Банковская карта онлайн | Мин. сумма: 100 RUB | Комиссия: 2.0%
URL: https://securepay.ru, Карта: 2200-****-****-1234, Держатель: Иван Иванов

--- Проверка дополнительных методов ---
Электронный чек отправлен на почту: student@mail.ru
Оформлен возврат 150 RUB на карту 2200-****-****-1234 (Иван Иванов)
Платеж по способу 'Банковская карта онлайн' отменен.

--- Взаимодействие объектов с классом Order ---

Попытка оплаты заказа №101 на сумму 1000 руб.
Успешно: платеж на сумму 1000 RUB (комиссия: 20.00 RUB) проведен через 'Банковская карта онлайн'.

Попытка оплаты заказа №102 на сумму 30 руб.
Ошибка: сумма 30 RUB меньше минимальной (100 RUB) для 'Банковская карта онлайн'.
